In [ ]:
import numpy as np

from scipy.stats import beta, lognorm
from scipy.optimize import minimize
from scipy.interpolate import PchipInterpolator

In [1]:
def calculate_pit_lognormal(mu, sigma, y):

    pit = lognorm.cdf(
        y,
        s=sigma,
        scale=np.exp(mu)
    )

    return pit 


In [ ]:
class PITRecalibration:
    """
    PIT-based forecast recalibration following Rumack et al. (2022).

    Predictive distribution:
        LogNormal(mu, sigma)

    Recalibration:
        F*(y) = G(F(y))

    where G is estimated from historical PIT values using:

        1. Null calibration
        2. Parametric Beta calibration
        3. Nonparametric calibration

    The three methods are combined using weights estimated
    by maximizing the log score.
    """

    def __init__(
        self,
        n_grid=10001,
        eps=1e-8
    ):
        self.n_grid = n_grid
        self.eps = eps

        self.pits = None

        self.beta_a = None
        self.beta_b = None

        self.weights = None

        self._G_np_x = None
        self._G_np_y = None

    # ==========================================================
    # FIT
    # ==========================================================

    def fit(
        self,
        pits,
        optimize_weights=True
    ):
        """
        Fit PIT recalibration.

        Parameters
        ----------
        pits : array-like
            Historical PIT values.

        optimize_weights : bool
            Estimate ensemble weights by maximizing log score.
        """

        pits = np.asarray(pits, dtype=float)

        pits = pits[np.isfinite(pits)]

        pits = np.clip(
            pits,
            self.eps,
            1 - self.eps
        )

        if len(pits) < 5:
            raise ValueError(
                "Too few PIT observations."
            )

        self.pits = pits

        # ------------------------------------------------------
        # Beta calibration
        # ------------------------------------------------------

        self._fit_beta()

        # ------------------------------------------------------
        # Non-parametric calibration
        # ------------------------------------------------------

        self._fit_nonparametric()

        # ------------------------------------------------------
        # Ensemble weights
        # ------------------------------------------------------

        if optimize_weights:
            self.weights = self._fit_weights()
        else:
            self.weights = np.array([
                1 / 3,
                1 / 3,
                1 / 3
            ])

        return self

    # ==========================================================
    # BETA
    # ==========================================================

    def _fit_beta(self):

        pits = self.pits

        def neg_loglik(theta):

            a = np.exp(theta[0])
            b = np.exp(theta[1])

            return -np.sum(
                beta.logpdf(
                    pits,
                    a,
                    b
                )
            )

        result = minimize(
            neg_loglik,
            x0=np.log([1., 1.]),
            method="BFGS"
        )

        if not result.success:
            raise RuntimeError(
                "Beta optimization failed."
            )

        self.beta_a, self.beta_b = np.exp(
            result.x
        )

    # ==========================================================
    # NON-PARAMETRIC G
    # ==========================================================

    def _fit_nonparametric(self):

        pits = np.sort(self.pits)

        n = len(pits)

        # Empirical PIT CDF
        #
        # G(p) = 1/n sum I(PIT_i <= p)

        G = np.arange(1, n + 1) / n

        # Add boundaries
        x = np.concatenate([
            [0.0],
            pits,
            [1.0]
        ])

        y = np.concatenate([
            [0.0],
            G,
            [1.0]
        ])

        # Duplicate PIT values can occur
        # Remove duplicate x values.
        x_unique, idx = np.unique(
            x,
            return_index=True
        )

        y_unique = y[idx]

        self._G_np_x = x_unique
        self._G_np_y = y_unique

    # ==========================================================
    # NON-PARAMETRIC G
    # ==========================================================

    def G_nonparametric(self, p):

        p = np.asarray(
            p,
            dtype=float
        )

        p = np.clip(
            p,
            0,
            1
        )

        return np.interp(
            p,
            self._G_np_x,
            self._G_np_y
        )

    # ==========================================================
    # BETA G
    # ==========================================================

    def G_beta(self, p):

        return beta.cdf(
            p,
            self.beta_a,
            self.beta_b
        )

    # ==========================================================
    # NULL G
    # ==========================================================

    @staticmethod
    def G_null(p):

        return np.asarray(p)

    # ==========================================================
    # COMPONENT CDFs
    # ==========================================================

    def component_G(self, p):

        return np.column_stack([
            self.G_nonparametric(p),
            self.G_beta(p),
            self.G_null(p)
        ])

    # ==========================================================
    # ENSEMBLE G
    # ==========================================================

    def G(self, p):

        components = self.component_G(p)

        return np.sum(
            components * self.weights,
            axis=1
        )

    # ==========================================================
    # FIT ENSEMBLE WEIGHTS
    # ==========================================================

    def _fit_weights(self):

        pits = self.pits

        # ------------------------------------------------------
        # Estimate density corresponding to each G
        #
        # The log score is evaluated at the observed PITs.
        # ------------------------------------------------------

        density_beta = beta.pdf(
            pits,
            self.beta_a,
            self.beta_b
        )

        # ------------------------------------------------------
        # Nonparametric density
        #
        # Piecewise derivative of empirical CDF.
        # To avoid zero density we use a small floor.
        # ------------------------------------------------------

        density_np = self._nonparametric_density(
            pits
        )

        # ------------------------------------------------------
        # Null density
        # ------------------------------------------------------

        density_null = np.ones_like(
            pits
        )

        densities = np.column_stack([
            density_np,
            density_beta,
            density_null
        ])

        # ------------------------------------------------------
        # Optimize:
        #
        # maximize
        #
        # sum log(
        #   w1 f_np +
        #   w2 f_beta +
        #   w3 f_null
        # )
        #
        # subject to:
        #
        # wi >= 0
        # sum wi = 1
        # ------------------------------------------------------

        def objective(w):

            mixture = (
                w[0] * densities[:, 0]
                + w[1] * densities[:, 1]
                + w[2] * densities[:, 2]
            )

            mixture = np.maximum(
                mixture,
                self.eps
            )

            return -np.sum(
                np.log(mixture)
            )

        result = minimize(
            objective,
            x0=np.array([
                1 / 3,
                1 / 3,
                1 / 3
            ]),
            method="SLSQP",
            bounds=[
                (0, 1),
                (0, 1),
                (0, 1)
            ],
            constraints={
                "type": "eq",
                "fun": lambda w: np.sum(w) - 1
            }
        )

        if not result.success:
            return np.array([
                1 / 3,
                1 / 3,
                1 / 3
            ])

        return result.x

    # ==========================================================
    # NONPARAMETRIC DENSITY
    # ==========================================================

    def _nonparametric_density(self, x):

        x_grid = self._G_np_x
        y_grid = self._G_np_y

        density = np.zeros_like(
            x,
            dtype=float
        )

        for i, value in enumerate(x):

            idx = np.searchsorted(
                x_grid,
                value,
                side="right"
            ) - 1

            idx = np.clip(
                idx,
                0,
                len(x_grid) - 2
            )

            dx = (
                x_grid[idx + 1]
                - x_grid[idx]
            )

            dy = (
                y_grid[idx + 1]
                - y_grid[idx]
            )

            if dx > 0:
                density[i] = dy / dx

        return np.maximum(
            density,
            self.eps
        )

    # ==========================================================
    # INVERSE G
    # ==========================================================

    def G_inverse(self, q):

        q = np.asarray(
            q,
            dtype=float
        )

        q = np.clip(
            q,
            self.eps,
            1 - self.eps
        )

        grid = np.linspace(
            self.eps,
            1 - self.eps,
            self.n_grid
        )

        G_grid = self.G(
            grid
        )

        # Numerical monotonicity
        G_grid = np.maximum.accumulate(
            G_grid
        )

        G_unique, idx = np.unique(
            G_grid,
            return_index=True
        )

        grid_unique = grid[idx]

        inverse = PchipInterpolator(
            G_unique,
            grid_unique
        )

        return np.clip(
            inverse(q),
            self.eps,
            1 - self.eps
        )

    # ==========================================================
    # RECALIBRATE LOGNORMAL
    # ==========================================================

    def recalibrate_lognormal(
        self,
        mu,
        sigma,
        probabilities
    ):
        """
        Recalibrate LogNormal quantiles.

        Q*(p) = F^{-1}(G^{-1}(p))
        """

        probabilities = np.asarray(
            probabilities,
            dtype=float
        )

        # New probability levels in original distribution
        p_original = self.G_inverse(
            probabilities
        )

        # Inverse LogNormal CDF
        quantiles = lognorm.ppf(
            p_original,
            s=sigma,
            scale=np.exp(mu)
        )

        return quantiles

    # ==========================================================
    # SAMPLE RECALIBRATED DISTRIBUTION
    # ==========================================================

    def sample_lognormal(
        self,
        mu,
        sigma,
        n_samples=1000,
        random_state=None
    ):
        """
        Generate samples from the recalibrated distribution.

        Uses:

            U ~ Uniform(0,1)

            P = G^{-1}(U)

            Y = F^{-1}(P)
        """

        rng = np.random.default_rng(
            random_state
        )

        u = rng.uniform(
            self.eps,
            1 - self.eps,
            size=n_samples
        )

        p = self.G_inverse(u)

        samples = lognorm.ppf(
            p,
            s=sigma,
            scale=np.exp(mu)
        )

        return samples

Recalibração: 

In [ ]:
df["pit"] = lognorm.cdf(
    df["y"],
    s=df["sigma"],
    scale=np.exp(df["mu"])
)

recalibrator = PITRecalibration()

recalibrator.fit(
    df["pit"].values
)

In [ ]:
5. Gerando samples

Aqui entra o que você pediu especificamente.

samples = recalibrator.sample_lognormal(
    mu=mu,
    sigma=sigma,
    n_samples=1000,
    random_state=42
)